<a href="https://colab.research.google.com/github/eduardozurek/ec2talleres/blob/main/explicaci%C3%B3n_de_perf_y_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

perf es una herramienta de análisis de rendimiento (performance profiling) para Linux. Permite observar cómo un programa utiliza el procesador y otros recursos del sistema mientras se ejecuta.

En una clase de rendimiento de sistemas computacionales, perf es especialmente útil porque conecta conceptos teóricos como tiempo de CPU, ciclos, instrucciones, IPC, fallos de caché, cambios de contexto y migraciones entre cores con mediciones reales.

1. ¿Qué es exactamente perf?

perf es una interfaz de usuario para el subsistema de performance events del kernel de Linux. Puede obtener información de varias fuentes:

* Hardware Performance Monitoring Counters (PMU): ciclos, instrucciones, referencias y fallos de caché, branches, branch misses, etc.

* Contadores del kernel: cambios de contexto, migraciones entre CPU, page faults.

* Eventos de software y tracepoints: actividad interna del kernel.

Por ejemplo:

perf stat ./programa

ejecuta programa y presenta estadísticas de rendimiento.

2. perf stat: el comando más útil para comenzar

Por ejemplo:

perf stat python3 programa.py

podría producir:

 Performance counter stats for 'python3 programa.py':

       981.65 msec task-clock
          414      context-switches
           15      cpu-migrations
         7552      page-faults
   2,500,000,000   cycles
   4,000,000,000   instructions
      50,000,000   branches
       2,000,000   branch-misses

       0.915 seconds time elapsed

Cada contador describe una dimensión diferente del comportamiento del programa.

|Métrica	|Qué mide |
|----|----|
|task-clock	|Tiempo acumulado usando CPU |
|context-switches	|Cambios de contexto |
|cpu-migrations	|Veces que un proceso/thread cambia de CPU/core
|page-faults	|Accesos que requieren intervención del sistema de memoria virtual
|cycles	|Ciclos de CPU consumidos
|instructions	|Instrucciones ejecutadas
|branches	|Instrucciones de salto ejecutadas
|branch-misses	|Predicciones de salto incorrectas

3. Una medición particularmente importante: IPC

Si tenemos:

cycles          2,500,000,000
instructions    4,000,000,000

podemos calcular:

$$ IPC=\frac{Instructions}{Cycles} $$

Por tanto:

$$ IPC=\frac{4.0\times10^9}{2.5\times10^9}=1.6 $$

Es decir, el procesador retiró en promedio 1.6 instrucciones por ciclo durante la ejecución.

El inverso es CPI:

$$ CPI=\frac{Cycles}{Instructions} $$

por lo que:

$$ CPI=\frac{1}{IPC}=0.625 $$

En procesadores modernos superscalares es perfectamente posible tener CPI < 1, porque pueden completar varias instrucciones por ciclo.

4. perf y sistemas multicore

Aquí perf se vuelve particularmente interesante.

Supongamos:

981.65 msec task-clock
0.91565 seconds time elapsed

Podemos calcular aproximadamente:

$$ CPU\ utilization = \frac{task\ clock}{elapsed\ time} $$ $$ =\frac{0.98165}{0.91565}=1.072 $$

o:

$$ 107.2\% $$

Esto significa aproximadamente 1.072 CPUs lógicas utilizadas en promedio.

Un programa estrictamente single-threaded debería estar cerca de:

$$ 1.0\ CPU = 100\% $$

mientras que un programa paralelo podría mostrar, por ejemplo:

3.82 CPUs utilized

lo que indica que durante su ejecución utilizó, en promedio, el equivalente a 3.82 CPUs lógicas simultáneamente.

Esto es especialmente relevante para sus ejemplos de multiplicación de matrices con NumPy/BLAS, porque el programa Python puede parecer secuencial, pero BLAS puede crear múltiples threads internamente.

5. Podemos seleccionar eventos específicos

No tenemos que medir todo. Por ejemplo:

perf stat -e task-clock,context-switches,cpu-migrations,page-faults ./programa

O, cuando el entorno permite acceder a los contadores hardware:

perf stat -e cycles,instructions,branches,branch-misses ./programa

Para memoria caché:

perf stat -e cache-references,cache-misses ./programa

Esto permite construir experimentos controlados para la clase.

6. perf stat no es todo perf

perf incluye varias herramientas. Las tres más importantes para comenzar son:

perf stat: ¿Cuántos recursos utilizó?

perf record: ¿Dónde se consumió el tiempo?

perf report: ¿Qué funciones fueron responsables?

Por ejemplo:

perf record ./programa

genera un perfil de ejecución. Después:

perf report

permite identificar qué funciones consumieron mayor porcentaje de CPU.

También existe:

perf top

que funciona conceptualmente como top, pero muestra qué funciones están consumiendo CPU en tiempo real.

7. Diferencia entre time y perf

Esto es importante para su clase porque son herramientas complementarias.

/usr/bin/time -v python3 programa.py

es excelente para medir:

elapsed time,
user time,
system time,
maximum resident set size,
page faults,
context switches.

Mientras que:

perf stat python3 programa.py

puede agregar información microarquitectónica como:

ciclos,
instrucciones,
IPC,
caché,
branches,
branch misses.

Podemos verlo como dos niveles:

                 Programa
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
       time                 perf
          │                   │
          ▼                   ▼
 Tiempo y memoria       CPU / microarquitectura
 elapsed time           cycles
 user time              instructions
 system time            IPC
 max RSS                cache misses
 page faults            branch misses

Por eso, para los experimentos que está preparando en GitHub Codespaces, una combinación muy didáctica sería medir el mismo programa primero con:

/usr/bin/time -v python3 programa.py

y después con:

perf stat python3 programa.py

Hay una salvedad importante: en Codespaces, contenedores, WSL2 y algunas máquinas virtuales, ciertos contadores hardware pueden no estar expuestos al entorno invitado. Por eso pueden aparecer cycles, instructions o branches como <not supported>, mientras que task-clock, context-switches, cpu-migrations y page-faults sí funcionan. Eso no significa necesariamente que perf esté mal instalado; puede ser una limitación de virtualización o permisos.-

Hay bastante solapamiento entre /usr/bin/time -v y perf stat, pero time -v proporciona algunas métricas importantes que perf stat normalmente no entrega directamente, especialmente relacionadas con memoria y E/S.

La diferencia conceptual es:

time caracteriza el consumo de recursos del proceso desde la perspectiva del sistema operativo.
perf caracteriza el comportamiento de rendimiento del proceso, incluyendo eventos del kernel y, cuando están disponibles, contadores hardware del procesador.

Comparación

| Información                             | `/usr/bin/time -v` | `perf stat` |
| --------------------------------------- | :----------------: | :---------: |
| Tiempo transcurrido (*elapsed*)         |          ✓         |      ✓      |
| User time                               |          ✓         |      ✓      |
| System time                             |          ✓         |      ✓      |
| % CPU                                   |          ✓         |  Calculable |
| Context switches                        |          ✓         |      ✓      |
| Page faults                             |          ✓         |      ✓      |
| CPU migrations                          |          —         |      ✓      |
| **Maximum Resident Set Size (Max RSS)** |        **✓**       |      —      |
| **Average resident set size**           |        **✓**       |      —      |
| **File-system inputs**                  |        **✓**       |      —      |
| **File-system outputs**                 |        **✓**       |      —      |
| **Socket messages**                     |        **✓**       |      —      |
| **Signals delivered**                   |        **✓**       |      —      |
| Cycles                                  |          —         |      ✓      |
| Instructions                            |          —         |      ✓      |
| IPC                                     |          —         |      ✓      |
| Cache misses                            |          —         |      ✓      |
| Branches                                |          —         |      ✓      |
| Branch misses                           |          —         |      ✓      |

La diferencia que considero más importante para sus experimentos con matrices es Max RSS.

Por ejemplo:

/usr/bin/time -v python3 matrices.py

puede mostrar:

Maximum resident set size (kbytes): 11427356

Esto indica aproximadamente el máximo de memoria RAM residente alcanzado por el proceso durante su ejecución.

perf stat estándar no le proporciona directamente ese pico de memoria.

Hay otra diferencia importante con los page faults

time -v los separa:

Major (requiring I/O) page faults: 10
Minor (reclaiming a frame) page faults: 7552

mientras que un perf stat típico puede mostrar:

page-faults

como un total. Con perf se pueden solicitar eventos más específicos dependiendo del sistema, pero time -v los presenta directamente y de forma muy conveniente.

Para su clase, usaría ambos

Para comparar, por ejemplo, multiplicación de matrices en Python/NumPy frente a C, ejecutaría:

/usr/bin/time -v python3 matrices.py

para estudiar principalmente:

$$ T_{elapsed},\quad T_{user},\quad T_{system},\quad MaxRSS,\quad PageFaults $$

y:

perf stat python3 matrices.py

para estudiar:

$$ CPU\ utilization,\quad cycles,\quad instructions,\quad IPC,\quad cache,\quad branches $$

Por tanto, perf no reemplaza completamente a time. Si quiere estudiar simultáneamente tiempo + CPU + memoria, /usr/bin/time -v sigue siendo muy útil, sobre todo por Maximum Resident Set Size. Para estudiar microarquitectura, perf es mucho más potente.

**¿Cómo puedo saber qué fuentes tengo disponibles?**

La forma más directa es pedirle al propio perf que enumere los eventos disponibles en ese sistema concreto:

perf list

Ese comando es especialmente importante porque lo disponible depende del procesador, kernel, permisos y entorno de virtualización. En Codespaces/VMs, por ejemplo, puede haber eventos que perf conoce pero cuyo contador hardware no está expuesto realmente.

1. Ver todos los eventos

Ejecute:

perf list

Obtendrá categorías similares a:

List of pre-defined events:

  cpu-cycles OR cycles
  instructions
  cache-references
  cache-misses
  branch-instructions OR branches
  branch-misses

  alignment-faults
  context-switches
  cpu-migrations
  page-faults
  ...

Para una primera inspección puede ser más cómodo:

perf list | less

y salir con q.

2. Consultar por categorías

Para los eventos hardware/PMU, pruebe:

perf list hardware

Normalmente son de especial interés:

cycles
instructions
cache-references
cache-misses
branches
branch-misses

Puede comprobarlos explícitamente:

perf stat -e cycles,instructions,cache-references,cache-misses,branches,branch-misses ls

Si están realmente disponibles, obtendrá números. Si su entorno no expone esos contadores, puede aparecer:

<not supported>

Esto último es relevante: que un evento aparezca en perf list no garantiza que la VM o contenedor permita medirlo.

3. Eventos software/kernel

Puede consultar:

perf list software

Entre los más útiles están:

task-clock
context-switches
cpu-migrations
page-faults

Y probarlos directamente:

perf stat -e task-clock,context-switches,cpu-migrations,page-faults ls

En entornos virtualizados estos suelen estar disponibles incluso cuando cycles e instructions no lo están.

4. Tracepoints del kernel

Para ver los tracepoints:

perf list tracepoint

Puede haber miles, por lo que normalmente conviene filtrar:

perf list tracepoint | less

o buscar una familia concreta:

perf list 'sched:*'

Por ejemplo, los relacionados con el scheduler:

sched:sched_switch
sched:sched_wakeup
sched:sched_process_exec
sched:sched_process_exit
...

También puede buscar system calls:

perf list 'syscalls:*'

Estos eventos permiten estudiar actividad interna del kernel con mucho más detalle que perf stat usando solamente sus eventos predeterminados.

5. PMUs disponibles en Linux

También puede mirar directamente lo que el kernel expone:

ls /sys/bus/event_source/devices/

Podría encontrar algo como:

breakpoint
cpu
kprobe
msr
software
tracepoint
uprobe

La entrada:

cpu

es particularmente importante porque representa la PMU del procesador que Linux tiene disponible.

Puede inspeccionarla con:

ls /sys/bus/event_source/devices/cpu/

y:

ls /sys/bus/event_source/devices/cpu/events/

si ese directorio existe.

6. Para su entorno, haría esta prueba

En la máquina donde está trabajando ejecute consecutivamente:

echo "=== PMUs ==="
ls /sys/bus/event_source/devices/

echo "=== Hardware ==="
perf list hardware

echo "=== Software ==="
perf list software

echo "=== Tracepoints scheduler ==="
perf list 'sched:*'

Después haga una prueba real, que es la verificación definitiva:



perf stat \ \
  -e task-clock,context-switches,cpu-migrations,page-faults,cycles,instructions,branches,branch-misses \ \
  ls



Si obtiene algo como:

1.12 msec task-clock
2    context-switches
0    cpu-migrations
120  page-faults

<not supported> cycles
<not supported> instructions
<not supported> branches
<not supported> branch-misses

la interpretación sería:

### Eventos de rendimiento disponibles en Linux con `perf`

| Fuente | Evento / Métrica | Disponibilidad |
|---|---|:---:|
| **Software / Kernel** | `task-clock` | ✅ Disponible |
| | `context-switches` | ✅ Disponible |
| | `cpu-migrations` | ✅ Disponible |
| | `page-faults` | ✅ Disponible |
| **Hardware / PMU** | `cycles` | ❌ No expuesto |
| | `instructions` | ❌ No expuesto |
| | `branches` | ❌ No expuesto |
| | `branch-misses` | ❌ No expuesto |
| **Tracepoints** | Eventos del kernel | ⚠️ Depende del kernel y los permisos |


Así que para determinar qué puede usar en sus prácticas, distinguiría entre “perf conoce el evento” (perf list) y “mi entorno realmente puede medirlo” (perf stat -e ... comando). La segunda prueba es la que finalmente importa.

**¿Cómo puedo enviar la salida de perf list a un archivo de texto?**

Puedes usar la redirección estándar de Linux:

perf list > perf_list.txt

Esto crea perf_list.txt en el directorio actual.

Para verificarlo:

ls -lh perf_list.txt

y para verlo:

less perf_list.txt

Si quieres agregar la salida a un archivo existente en vez de sobrescribirlo:

perf list >> perf_list.txt

También puedes guardar categorías específicas:

perf list hardware > hardware.txt
perf list software > software.txt
perf list tracepoint > tracepoints.txt

Para tus prácticas, una opción útil es:

perf list > perf_eventos_disponibles.txt

Luego ese archivo te permite comparar fácilmente qué eventos reporta perf en Codespaces, WSL2, una VM y una máquina Linux física.